In [1]:
import torch
from google.colab import drive
import os

drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aiffel
os.chdir('/content/drive/MyDrive/aiffel')
os.environ["HF_HOME"] = "/root/.cache/huggingface"

print("GPU 사용 가능 여부:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("연결된 GPU 이름:", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU 사용 가능 여부: True
연결된 GPU 이름: Tesla T4


# 오픈소스 LLM 비교 분석과 앱 설계

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 들어가며

이 레슨에서는 VRAM 계산기를 만들고 후보 모델을 실제로 올려 한국어 업무 태스크로 비교합니다. 마지막에 다음 노드(사내 데이터 기반 LLM 파인튜닝 프로젝트)에서 만들 프로젝트의 설계서를 씁니다.

이론 레슨 6.2절의 **자르는 네 단계**(경계, 태스크, 라이선스, 메모리)를 코드로 옮깁니다. 첫 단계인 경계는 오늘 **"우리 GPU 안에서만 처리한다"로 고정**하고 시작합니다. 그래서 후보는 전부 오픈 웨이트입니다.

#### 실행 환경

이 레슨은 **GPU 런타임**이 필요합니다. 실습 환경 GPU(T4, VRAM 16GB)에서 처음부터 끝까지 돌아갑니다. 런타임 유형이 GPU인지 확인하고 시작하십시오.

여러 모델을 동시에 올릴 VRAM이 없으므로 모델은 **한 번에 하나만** 올려 재고 내리기를 반복합니다. 내리는 코드가 제대로 동작하는지는 Step 3에서 숫자로 확인합니다.

**키는 없어도 됩니다.** 게이트가 걸린 저장소 두 곳은 승인 없이 쓸 수 있는 미러로 대체하고 미러도 막히면 그 모델만 건너뛰고 **건너뛴 이유를 표에 남깁니다.** 원본 저장소를 그대로 쓰고 싶으면 Step 0의 `login()` 안내를 보십시오.

PC나 사내 서버에서 해야 하는 일에는 🖥 **로컬 작업** 표시를 붙였습니다. 실습 환경에서는 읽고 넘어가고 수료 후 사내에 옮길 때 다시 보면 됩니다.

| 구간 | 하는 일 | 걸리는 시간 |
|---|---|---|
| Step 0 | 라이브러리 설치 | 1~2분 |
| Step 1 | 설정 파일 4개 내려받기 (수백 바이트씩) | 10초 안쪽 |
| Step 2 | 평가 세트와 채점기 검산 (모델 없이 도는 구간) | 즉시 |
| Step 3 | 3B 두 개를 차례로 올려 태스크 5종 측정 | 8~12분 |
| 트랙 2 | 7B와 3B를 한 번씩 올림 | 10~15분 |
| 트랙 5 | 같은 모델을 정밀도 셋으로 올림 | 12~18분 |

Step 3과 트랙 2, 트랙 5는 모델을 내려받고 올리는 동안 출력이 없어 **멈춘 것처럼 보입니다.** 셀 왼쪽의 회전 표시가 돌고 있으면 정상입니다.

실습 셀(코드가 비어 있고 바로 아래 '정답 보기'가 붙은 셀)의 저장된 출력은 정답 코드를 넣고 실행한 결과입니다. 코드를 채우기 전에 그대로 실행하면 다른 결과가 나옵니다.

#### 실습에 쓰는 모델 판본

이론 레슨 1.2절의 계열 지도는 Qwen3.5와 3.6, Llama 4, Gemma 4처럼 2026년 9월 현행 판본을 적고 있습니다. 그런데 이 레슨의 후보는 한두 세대 앞의 **Qwen2.5, Llama 3.2, Gemma 2**입니다.

**앞 세대를 쓰는 이유**

- **작은 판이 있어야 실습 환경 GPU 한 장에 올라갑니다.** 현행 계열의 최소 크기가 실습 GPU를 넘는 경우가 있습니다  
- **설정 파일 구조가 검증된 판본이어야 계산기의 검산이 성립합니다.** 새 판본은 설정 키 이름이 바뀌어 계산식이 조용히 틀릴 수 있습니다  
- **게이트와 미러 상황이 확인된 저장소**여야 승인 없이 수업이 진행됩니다

그래서 오늘 익힐 것은 모델 이름이 아니라 **절차**입니다. 이름은 반년이면 바뀌어도 설정 파일을 읽어 메모리를 계산하고 우리 태스크로 재는 순서는 그대로입니다. 사내에서 실제로 고를 때는 같은 코드에 현행 판본의 저장소 이름만 넣으면 됩니다.

## Step 0 : 환경 준비

In [2]:
# 메이저 버전에 상한을 둡니다. 상한이 없으면 다음 메이저 판이 나오는 날
# 이 레슨이 임포트 단계에서 멈춥니다
%pip install -q "transformers>=5.17,<6" "accelerate>=1.15,<2" "bitsandbytes>=0.50,<0.51"

# 설치가 끝난 뒤 pip이 의존성 충돌을 보고하면 [런타임] - [세션 다시 시작]을 한 번 누르고
# 이 셀 아래부터 다시 실행하세요. transformers를 고정하면 huggingface-hub가 함께
# 내려가면서 충돌 메시지가 나올 수 있습니다
#
# 🖥 로컬 작업 안내 : 게이트가 걸린 저장소(meta-llama, google/gemma)를 원본 그대로 쓰려면
#    승인과 토큰이 필요합니다. 순서는 이렇습니다
#      1) 모델 카드 페이지에서 약관에 동의해 승인을 받습니다 (몇 분에서 며칠)
#      2) 실습 환경 왼쪽의 열쇠 아이콘(Secrets)에 HF_TOKEN 을 넣고 아래 두 줄의 주석을 풉니다
# from google.colab import userdata
# from huggingface_hub import login; login(userdata.get("HF_TOKEN"))
#
#    사내 폐쇄망에 올릴 때는 이 승인을 **구축 전에** 받아 가중치를 미리 받아 두어야 합니다.
#    이론 레슨 4.4절이 말한 일정 문제가 여기서 생깁니다

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 64.9 MB/s eta 0:00:00:00:0100:01


In [3]:
import torch, json, time, gc, re
import numpy as np
from importlib.metadata import version, PackageNotFoundError
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import logging as hub_logging
import logging

# 라이브러리의 안내 경고를 끕니다. 켜 두면 Step 3과 트랙의 출력에서 측정 결과가 경고 사이에 묻힙니다
hf_logging.set_verbosity_error()         # 생성 설정 안내(max_length 등)
hub_logging.set_verbosity_error()        # '토큰 없이 요청 중' 안내
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)   # 8bit 곱셈의 dtype 변환 경고

print(f"GPU : {torch.cuda.get_device_name(0)}")
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM : {TOTAL_VRAM:.1f} GB")


def pkgver(dist):
    """배포판 이름으로 설치된 버전을 읽습니다"""
    try:
        return version(dist)
    except PackageNotFoundError:
        return "미설치"


# 설치된 버전을 기록해둡니다. 문제가 생기면 이 숫자부터 확인합니다
print("\n[설치된 버전]")
for d in ["transformers", "accelerate", "bitsandbytes", "torch"]:
    print(f"  {d:16s} {pkgver(d)}")

GPU : Tesla T4
VRAM : 15.6 GB

[설치된 버전]
  transformers     5.18.0
  accelerate       1.15.0
  bitsandbytes     0.50.2
  torch            2.11.0+cu130


## Step 1 : VRAM 계산기 만들기

이론 레슨의 세 가지 소모원을 코드로 옮깁니다. 이 계산은 모델을 내려받기 전에 **설정 파일만 읽어서** 할 수 있습니다. 수십 GB를 받아 본 뒤에 안 올라간다는 것을 알면 너무 늦습니다.

In [4]:
# 후보 목록. 게이트가 걸린 저장소에는 승인 없이 받을 수 있는 미러를 함께 적어둡니다.
# 미러는 같은 가중치를 다시 올린 재배포본입니다. 파일은 같아도 **라이선스는 원본을 따릅니다**
CANDIDATES = [
    {"id": "Qwen/Qwen2.5-3B-Instruct", "mirror": None},
    {"id": "Qwen/Qwen2.5-7B-Instruct", "mirror": None},
    {"id": "meta-llama/Llama-3.2-3B-Instruct", "mirror": "unsloth/Llama-3.2-3B-Instruct"},
    {"id": "google/gemma-2-2b-it", "mirror": "unsloth/gemma-2-2b-it"},
]


def fetch_config(entry):
    """설정 파일을 받아옵니다. config만 내려받으므로 수백 바이트 수준입니다.

    실패를 예외로 터뜨리지 않고 **상태를 돌려줍니다.** 후보 하나가 막혔다고
    비교표가 한 줄로 줄어들면 안 되기 때문입니다.
    돌려주는 것 : (설정 또는 None, 실제로 받은 저장소 또는 None, 상태 한 줄)
    """
    try:
        return AutoConfig.from_pretrained(entry["id"]), entry["id"], "열림"
    except Exception as e:
        first = type(e).__name__          # 승인이 필요하면 여기서 걸립니다
    if not entry["mirror"]:
        return None, None, f"막힘 : {first}"
    try:
        cfg = AutoConfig.from_pretrained(entry["mirror"])
        return cfg, entry["mirror"], "게이트 -> 미러"
    except Exception as e:
        return None, None, f"막힘 : {first} / 미러도 {type(e).__name__}"


models = {}          # 후보 이름 -> {"cfg", "repo", "gate"}
for e in CANDIDATES:
    cfg, repo, gate = fetch_config(e)
    models[e["id"]] = {"cfg": cfg, "repo": repo, "gate": gate}
    used = f"  ->  {repo}" if repo and repo != e["id"] else ""
    print(f"[{gate:<14}] {e['id']}{used}")

# 뒤의 계산은 설정을 받은 후보로만 합니다. 못 받은 후보도 models에는 남아 있어
# 표에 "건너뜀"으로 찍힙니다
configs = {k: v["cfg"] for k, v in models.items() if v["cfg"] is not None}
print(f"\n설정 확보 {len(configs)} / {len(models)}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[열림            ] Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[열림            ] Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

[게이트 -> 미러     ] meta-llama/Llama-3.2-3B-Instruct  ->  unsloth/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/913 [00:00<?, ?B/s]

[게이트 -> 미러     ] google/gemma-2-2b-it  ->  unsloth/gemma-2-2b-it

설정 확보 4 / 4


상태 칸을 먼저 보십시오. **`게이트 -> 미러`로 찍힌 줄은 폴백이 실제로 동작했다는 뜻이고** 원본이 막힌 것을 감지하지 못하면 이 표시가 나오지 않습니다. 이론 레슨 4.4절은 게이트 여부가 기술적으로는 사소해도 일정에 영향을 주므로 표에 한 칸으로 넣으라고 했습니다.

미러도 막히면 그 후보는 `막힘`으로 남고 검산 표와 비교표에 **건너뛴 이유가 그대로 찍힙니다.** 표에서 줄이 조용히 사라지면 나중에 "그 모델은 왜 비교에서 빠졌지"라는 질문에 답할 수 없습니다.

In [5]:
# 설정에서 필요한 값을 읽습니다. 못 받은 후보는 사유와 함께 남깁니다
print(f"{'모델':<30} {'레이어':>6} {'hidden':>7} {'Q헤드':>6} {'KV헤드':>6} {'어휘':>7} {'게이트':>14}")
print("-" * 86)
for name, m in models.items():
    short = name.split("/")[-1]
    cfg = m["cfg"]
    if cfg is None:
        print(f"{short:<30} {'건너뜀 : ' + m['gate']:>50}")
        continue
    n_kv = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
    print(f"{short:<30} {cfg.num_hidden_layers:>6} {cfg.hidden_size:>7} "
          f"{cfg.num_attention_heads:>6} {n_kv:>6} {cfg.vocab_size:>7} {m['gate']:>14}")

모델                                레이어  hidden    Q헤드   KV헤드      어휘            게이트
--------------------------------------------------------------------------------------
Qwen2.5-3B-Instruct                36    2048     16      2  151936             열림
Qwen2.5-7B-Instruct                28    3584     28      4  152064             열림
Llama-3.2-3B-Instruct              28    3072     24      8  128256      게이트 -> 미러
gemma-2-2b-it                      26    2304      8      4  256000      게이트 -> 미러


Q헤드와 KV헤드 수를 비교해 보십시오. 차이가 큰 모델은 이론 레슨 2.4절의 GQA를 강하게 적용한 것이고 그만큼 KV 캐시가 작습니다.

계산기를 만들기 전에 **계산기는 먼저 검산한다**는 원칙을 세웁니다. 모델 카드에 파라미터 수가 공개되어 있으므로 우리 계산이 그 숫자를 맞히는지부터 확인합니다. 검산 없는 계산기는 틀린 숫자를 자신 있게 알려 줍니다.

In [ ]:
# 실습 : VRAM 계산 함수를 완성하세요
# 세 가지를 더합니다 : 가중치 + KV 캐시 + 오버헤드(15%)

BYTES = {"fp32": 4, "fp16": 2, "int8": 1, "int4": 0.5}


def head_dim_of(cfg):
    """헤드 차원. 대부분 hidden/헤드수지만 config에 따로 적힌 모델이 있습니다"""
    return getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads


def max_ctx_of(cfg):
    """이 모델이 받을 수 있는 최대 길이. 숫자를 박지 않고 설정에서 읽어옵니다"""
    return getattr(cfg, "max_position_embeddings", None)


def split_params(cfg):
    """파라미터를 **4bit로 바뀌는 부분**과 **fp16으로 남는 부분**으로 나눠 반환합니다.

    나누는 이유가 중요합니다. bitsandbytes는 `nn.Linear`만 4bit로 바꾸고,
    임베딩 층과 출력 층(lm_head)은 fp16으로 남깁니다. 둘을 합쳐서 계산하면
    4bit 로드 용량을 크게 과소 추정합니다.
    """
    h = cfg.hidden_size
    L = cfg.num_hidden_layers
    inter = getattr(cfg, "intermediate_size", 4 * h)
    n_kv = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
    hd = head_dim_of(cfg)
    q_out = cfg.num_attention_heads * hd          # Q 투영의 출력 차원

    # 어텐션 : Q 투영 + K,V 투영(GQA로 축소) + O 투영
    attn = h * q_out + 2 * (h * n_kv * hd) + q_out * h
    # MLP : gate, up, down 세 개
    mlp = 3 * h * inter
    linear = L * (attn + mlp)                     # 양자화 대상

    # 출력 층이 임베딩과 가중치를 공유하는 모델(tie_word_embeddings)은 한 번만 셉니다
    n_matrix = 1 if getattr(cfg, "tie_word_embeddings", False) else 2
    embed = cfg.vocab_size * h * n_matrix         # fp16으로 남음

    return linear, embed


def estimate_params(cfg):
    """총 파라미터 수"""
    linear, embed = split_params(cfg)
    return linear + embed


def kv_cache_bytes(cfg, seq_len, batch_size, precision="fp16"):
    """KV 캐시 크기를 바이트로 반환하세요
    공식 : 2 x 레이어 x KV헤드 x 헤드차원 x 시퀀스 x 배치 x 정밀도바이트
    """
    layers = cfg.num_hidden_layers
    n_kv = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
    hd = head_dim_of(cfg)
    prec_bytes = BYTES[precision]

    return 2 * layers * n_kv * hd * seq_len * batch_size * prec_bytes


def estimate_vram(
    cfg, weight_precision="int4", seq_len=2048, batch_size=1, overhead=0.15
):
    """총 VRAM 추정치를 GB로 반환하세요

    overhead=0.15 는 계산에 안 잡히는 것들(활성값, 프레임워크 버퍼, 단편화) 몫입니다.
    뒤에서 GPU 전체의 90%까지만 쓴다는 규칙을 하나 더 겁니다. 둘을 곱하면
    1.15 / 0.9 = 약 1.28, 즉 계산값 위에 **28퍼센트 여유**가 됩니다.
    이론 레슨 2.5절이 권하는 20~40퍼센트 범위 안입니다
    """
    # 1. 모델 가중치(Bytes) : Linear(4bit 등) + Embed(FP16 유지)
    linear, embed = split_params(cfg)
    weight_bytes = (linear * BYTES[weight_precision]) + (embed * BYTES["fp16"])

    # 2. KV 캐시(Bytes) : 기본적으로 FP16으로 유지
    kv_bytes = kv_cache_bytes(
        cfg, seq_len=seq_len, batch_size=batch_size, precision="fp16"
    )

    # 3. 합산 후 15% 오버헤드 반영
    total_bytes = (weight_bytes + kv_bytes) * (1 + overhead)

    # 4. 바이트를 GB(GiB)로 변환 (1024^3으로 나눔)
    total_gb = total_bytes / (1024**3)

    return round(total_gb, 2)


for name, cfg in configs.items():
    print(f"{name} : {estimate_vram(cfg)} GB")

Qwen/Qwen2.5-3B-Instruct : 2.23 GB
Qwen/Qwen2.5-7B-Instruct : 5.95 GB
meta-llama/Llama-3.2-3B-Instruct : 2.6 GB
google/gemma-2-2b-it : 2.58 GB


### Step 1.5 : 계산기 검산

프라이빗 RAG 노드에서 4bit 로드 용량을 이야기할 때 "7B를 4bit로 줄이면 7 × 0.5 = 3.5GB"라는 계산이 왜 틀리는지 다뤘습니다. 여기서 그것을 코드로 확인합니다.

파라미터 수가 모델 카드와 맞는지, 4bit 가중치를 어떻게 쪼개야 하는지 두 가지를 검산합니다.

In [10]:
# 모델 카드에 공개된 파라미터 수 (단위 : 10억)
PUBLISHED = {
    "Qwen/Qwen2.5-3B-Instruct": 3.09,
    "Qwen/Qwen2.5-7B-Instruct": 7.62,
    "meta-llama/Llama-3.2-3B-Instruct": 3.21,
    "google/gemma-2-2b-it": 2.61,
}

print(f"{'모델':<24} {'추정':>7} {'공개':>7} {'오차':>7} {'tie':>6} {'head_dim':>9} {'최대길이':>9}")
print("-" * 76)
for name, m in models.items():
    short = name.split("/")[-1]
    cfg = m["cfg"]
    if cfg is None:
        print(f"{short:<24} {'건너뜀 : ' + m['gate']:>50}")
        continue
    total = estimate_params(cfg) / 1e9
    pub = PUBLISHED.get(name)
    tie = getattr(cfg, "tie_word_embeddings", False)
    err = f"{(total - pub) / pub:>6.1%}" if pub else "     -"
    print(f"{short:<24} {total:>7.2f} {pub if pub else 0:>7.2f} {err} "
          f"{str(tie):>6} {head_dim_of(cfg):>9} {str(max_ctx_of(cfg)):>9}")

모델                            추정      공개      오차    tie  head_dim      최대길이
----------------------------------------------------------------------------
Qwen2.5-3B-Instruct         3.09    3.09  -0.1%   True       128     32768
Qwen2.5-7B-Instruct         7.62    7.62  -0.1%  False       128     32768
Llama-3.2-3B-Instruct       3.21    3.21   0.1%   True       128    131072
gemma-2-2b-it               2.61    2.61   0.2%   True       256      8192


오차가 1퍼센트 안쪽이면 계산기를 믿을 수 있습니다.

걸리기 쉬운 함정이 있습니다.

- **`tie_word_embeddings`** : 이 값이 True인 모델은 출력 층이 임베딩과 가중치를 공유합니다. 임베딩 행렬을 두 번 세면 파라미터가 부풀고 어휘가 큰 모델일수록 심합니다. gemma-2-2b는 어휘가 25만 6천이라 두 번 세면 20퍼센트 넘게 과다 계상됩니다  
- **`head_dim`** : 대부분 hidden을 헤드 수로 나눈 값이지만 config에 따로 적어 두는 모델이 있습니다. gemma-2-2b는 hidden 2304에 헤드 8개인데 head_dim이 288이 아니라 256입니다. 나눗셈으로 구하면 KV 캐시가 12퍼센트 부풀고 Q 투영 크기도 틀립니다

두 함정 모두 **예외를 던지지 않고** 숫자만 조용히 틀립니다. 표의 오차 열이 없으면 발견할 방법이 없습니다.

최대 길이 열은 동시 사용자 수에 따른 VRAM 계산에서 씁니다. 시퀀스 길이는 우리가 마음대로 정하지 못합니다. **모델이 받을 수 있는 한계가 설정 파일에 적혀 있습니다.** 이 숫자를 코드에 직접 적지 말고 읽어 쓰십시오.

In [11]:
# 4bit 로드 시 가중치가 어떻게 쪼개지는지 봅니다
print(f"{'모델':<26} {'선형(4bit)':>11} {'임베+출력(fp16)':>16} {'합계':>8} {'단순계산':>9} {'차이':>7}")
print("-" * 84)
for name, cfg in configs.items():
    linear, embed = split_params(cfg)
    q = linear * BYTES["int4"] / 1e9          # 선형 계층만 4bit
    f = embed * BYTES["fp16"] / 1e9           # 임베딩과 lm_head는 fp16
    naive = (linear + embed) * BYTES["int4"] / 1e9   # 전부 4bit라고 잘못 가정
    print(f"{name.split('/')[-1]:<26} {q:>11.2f} {f:>16.2f} {q+f:>8.2f} {naive:>9.2f} "
          f"{(naive-(q+f))/(q+f):>6.0%}")

모델                            선형(4bit)      임베+출력(fp16)       합계      단순계산      차이
------------------------------------------------------------------------------------
Qwen2.5-3B-Instruct               1.39             0.62     2.01      1.54   -23%
Qwen2.5-7B-Instruct               3.26             2.18     5.44      3.81   -30%
Llama-3.2-3B-Instruct             1.41             0.79     2.20      1.61   -27%
gemma-2-2b-it                     1.01             1.18     2.19      1.31   -40%


`차이` 열은 "전부 4bit"라고 가정했을 때의 오차입니다. **값이 음수면 단순 계산이 그만큼 적게 잡았다는 뜻입니다.** **어휘가 크고 모델이 작을수록** 임베딩의 비중이 커져 적게 잡는 폭도 커집니다.

7B를 4bit로 올리면 **5.4GB 남짓**입니다. 프라이빗 RAG 노드에서 본 어림셈의 3.5GB는 물론 전부 4bit로 계산한 3.8GB보다도 큽니다. 여유를 계산할 때는 2GB 가까운 이 차이까지 넣어야 합니다.

<details>
<summary>정답 보기</summary>

```python
def kv_cache_bytes(cfg, seq_len, batch_size, precision="fp16"):
    n_kv = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
    return (2 * cfg.num_hidden_layers * n_kv * head_dim_of(cfg)
            * seq_len * batch_size * BYTES[precision])

def estimate_vram(cfg, weight_precision="int4", seq_len=2048, batch_size=1, overhead=0.15):
    linear, embed = split_params(cfg)
    # 선형 계층만 지정 정밀도로, 임베딩과 출력 층은 fp16으로 계산합니다
    weights = linear * BYTES[weight_precision] + embed * BYTES["fp16"]
    kv = kv_cache_bytes(cfg, seq_len, batch_size)          # KV 캐시는 보통 fp16
    total = (weights + kv) * (1 + overhead)
    return round(total / 1e9, 2)
```

**코드에서 볼 것**

첫째, **가중치를 한 덩어리로 곱하지 않았습니다.** `linear`에는 지정한 정밀도를, `embed`에는 fp16을 적용합니다. `weight_precision="fp16"`으로 부르면 둘 다 2바이트가 되어 결과가 같아지므로 이 분리는 양자화할 때만 차이를 만듭니다.

둘째, **KV 캐시는 fp16으로 계산합니다.** 가중치를 4bit로 양자화해도 KV 캐시는 기본적으로 fp16으로 유지됩니다. KV 캐시까지 양자화하는 기법이 있지만 별도 설정이 필요합니다.

**실행 순서** : 정답의 두 함수를 실습 셀의 `pass` 자리에 채워 넣고 그 셀을 다시 실행하십시오. 채우기 전에는 `None GB`가 찍히는 것이 정상이고 채운 뒤에는 정밀도별 비교 셀부터 순서대로 이어서 실행하십시오. Step 1.5의 검산 표에서 오차가 1퍼센트를 넘으면 다음으로 넘어가지 말고 `split_params`부터 다시 보십시오.

</details>

In [12]:
# 정밀도별 비교
print(f"{'모델':<30} {'int4':>8} {'int8':>8} {'fp16':>8}")
print("-" * 58)
for name, cfg in configs.items():
    row = [estimate_vram(cfg, p) for p in ["int4", "int8", "fp16"]]
    mark = ["O" if v < TOTAL_VRAM else "X" for v in row]
    print(f"{name.split('/')[-1]:<30} " + " ".join(f"{v:>6.1f}{m}" for v, m in zip(row, mark)))

모델                                 int4     int8     fp16
----------------------------------------------------------
Qwen2.5-3B-Instruct               2.2O    3.7O    6.7O
Qwen2.5-7B-Instruct               6.0O    9.4O   16.4X
Llama-3.2-3B-Instruct             2.6O    4.1O    7.1O
gemma-2-2b-it                     2.6O    3.7O    5.8O


In [13]:
# 이론 레슨 2.5절의 핵심 : 모델이 올라가는 것과 서비스가 되는 것은 다른 문제
# 동시 사용자 수에 따른 VRAM 변화를 봅니다

cfg = configs["Qwen/Qwen2.5-7B-Instruct"]
linear, embed = split_params(cfg)
w = (linear * BYTES["int4"] + embed * BYTES["fp16"]) / 1e9

# 전체 VRAM을 그대로 쓸 수는 없습니다. 프레임워크 버퍼와 단편화를 감안해 90%만 씁니다.
# estimate_vram의 overhead 15%와 이 90%를 곱하면 계산값 위에 약 28% 여유가 됩니다
USABLE = TOTAL_VRAM * 0.9

# 시퀀스 길이도 박지 않고 모델이 받을 수 있는 한계 안에서 고릅니다
SEQ = min(4096, max_ctx_of(cfg) or 4096)

print(f"Qwen2.5-7B (int4), 시퀀스 {SEQ} 기준")
print(f"  가중치      : {w:.2f} GB  (선형 4bit {linear*BYTES['int4']/1e9:.2f} + "
      f"임베딩/출력 fp16 {embed*BYTES['fp16']/1e9:.2f})")
print(f"  사용 가능   : {USABLE:.1f} GB  (전체 {TOTAL_VRAM:.1f} GB의 90%)\n")
print(f"{'동시 사용자':>12} {'VRAM(GB)':>10} {'KV 비중':>9} {'가능?':>8}")
print("-" * 44)
for b in [1, 4, 8, 16, 32, 64, 128]:
    v = estimate_vram(cfg, "int4", seq_len=SEQ, batch_size=b)
    kv = kv_cache_bytes(cfg, SEQ, b) / 1e9
    print(f"{b:>12} {v:>10.1f} {kv/v:>8.0%} {'O' if v < USABLE else 'X':>8}")

Qwen2.5-7B (int4), 시퀀스 4096 기준
  가중치      : 5.44 GB  (선형 4bit 3.26 + 임베딩/출력 fp16 2.18)
  사용 가능   : 14.1 GB  (전체 15.6 GB의 90%)

      동시 사용자   VRAM(GB)     KV 비중      가능?
--------------------------------------------
           1        6.1       4%        O
           4        6.8      14%        O
           8        7.8      24%        O
          16        9.8      38%        O
          32       13.9      54%        O
          64       21.9      69%        X
         128       38.0      79%        X


**KV 비중 열을 따라 읽어 보십시오.** 사용자 1명일 때는 전체의 몇 퍼센트에 불과하지만 128명이면 절반을 훌쩍 넘깁니다. 가중치는 고정값이라 사용자가 늘어도 그대로이고 KV 캐시만 선형으로 커집니다.

그래서 **가중치가 올라간다는 것과 서비스가 된다는 것은 다른 문제**입니다. 5.4GB 모델이 15.6GB GPU에 올라간다고 안심할 수 없습니다. 몇 명을 동시에 받을지가 실제 한계를 정합니다.

서빙 심화 주간에서 배우는 vLLM은 KV 캐시를 페이지 단위로 관리해 이 낭비를 줄입니다. 지금은 문제의 크기를 아는 것으로 충분합니다.

표에서 "가능?"이 X로 바뀌는 지점도 기억해 두십시오. 서빙 기초 주간의 마지막 날에 부하 시험으로 실제 한계를 측정하는데 **그때 여기서 계산한 숫자와 비교합니다.** 계산이 실측과 얼마나 맞는지가 용량 산정 능력의 핵심입니다.

In [14]:
# 실습 : 목표 동시 사용자 수를 만족하는 최대 모델 크기를 찾는 함수를 만드세요
# 우리 GPU에서 동시 사용자 8명, 시퀀스 4096을 처리하려면 어느 후보가 가능한가?

def feasible(cfg, target_users, seq_len=4096, precision="int4", margin=0.9):
    seq = min(seq_len, max_ctx_of(cfg) or seq_len)      # 모델 한계를 넘지 않게
    v = estimate_vram(cfg, precision, seq_len=seq, batch_size=target_users)
    return v <= TOTAL_VRAM * margin

print(f"{'모델':<30} {'1명':>6} {'4명':>6} {'8명':>6}")
print("-" * 52)
for name, cfg in configs.items():
    marks = ["O" if feasible(cfg, u) else "X" for u in [1, 4, 8]]
    print(f"{name.split('/')[-1]:<30} " + " ".join(f"{m:>6}" for m in marks))

모델                                 1명     4명     8명
----------------------------------------------------
Qwen2.5-3B-Instruct                 O      O      O
Qwen2.5-7B-Instruct                 O      O      O
Llama-3.2-3B-Instruct               O      O      O
gemma-2-2b-it                       O      O      O


<details>
<summary>정답 보기</summary>

```python
def feasible(cfg, target_users, seq_len=4096, precision="int4", margin=0.9):
    seq = min(seq_len, max_ctx_of(cfg) or seq_len)      # 모델 한계를 넘지 않게
    v = estimate_vram(cfg, precision, seq_len=seq, batch_size=target_users)
    return v <= TOTAL_VRAM * margin

print(f"{'모델':<30} {'1명':>6} {'4명':>6} {'8명':>6}")
print("-" * 52)
for name, cfg in configs.items():
    marks = ["O" if feasible(cfg, u) else "X" for u in [1, 4, 8]]
    print(f"{name.split('/')[-1]:<30} " + " ".join(f"{m:>6}" for m in marks))
```

활성값과 프레임워크 오버헤드에 여유를 남기려고 margin을 0.9로 두었습니다. 추정치를 그대로 믿고 100퍼센트까지 채우면 실제로는 메모리가 모자랍니다. 여기에 `estimate_vram`이 이미 얹은 15퍼센트가 더해져 **여유를 두 군데에 나눠 둔 셈**입니다. 어디에 얼마를 두었는지 적어 두는 것이 한 숫자로 맞추는 것보다 중요합니다.

**실행 순서** : 이 블록을 실습 셀 아래에 덧붙여 실행하십시오. `feasible`이 새로 정의되면서 표가 찍힙니다. 표의 O가 하나도 없으면 GPU가 작은 것이므로 `precision="int4"`인지부터 확인하십시오.

</details>

## Step 2 : 평가 세트 만들기

이론 레슨 6.3절에서 모델은 우리 업무 데이터로 직접 재서 고른다고 했습니다. 사내 업무를 흉내 낸 태스크 5종을 준비합니다.

각 태스크에는 답변 문자열을 받아 통과 여부를 돌려주는 **채점기**(`check`)가 붙습니다. 지금 쓰는 채점기는 **처음 떠오른 대로 쓴 것**입니다. Step 2.5에서 검산합니다.

In [15]:
# 사내 업무를 모사한 5종 태스크
tasks = [
    {
        "id": "T1_요약",
        "prompt": """다음 회의록을 3줄로 요약하세요. 결정 사항을 반드시 포함하세요.

[회의록]
9월 정기 개발회의. 참석 6명.
1. 신규 검색 기능 출시일을 10월 15일로 확정. QA 기간 2주 확보.
2. 레거시 API 폐기는 4분기로 연기. 외부 연동 고객 3곳 마이그레이션 미완료.
3. 인프라 비용 절감안 논의. GPU 인스턴스 사용률 40퍼센트로 낮아 축소 검토.
4. 다음 회의 10월 7일.""",
        "check": lambda s: ("10월 15일" in s or "10/15" in s) and len(s.split("\n")) <= 6,
        "desc": "정보 압축과 형식 준수",
    },
    {
        "id": "T2_추출",
        "prompt": """다음 문장에서 정보를 추출해 JSON으로만 출력하세요.
형식: {"고객사": "", "금액": 0, "납기": "", "담당자": ""}

"주식회사 한빛물산과 총 4,800만원 규모의 계약을 체결했으며, 납기는 2026년 11월 30일, 담당자는 영업2팀 김서준 과장입니다." """,
        "check": lambda s: "한빛물산" in s and ("4800" in s.replace(",", "") or "48000000" in s.replace(",", "")),
        "desc": "구조화 출력",
    },
    {
        "id": "T3_분류",
        "prompt": """다음 문의를 [기술지원], [영업], [환불], [기타] 중 하나로 분류하세요.
분류 결과만 대괄호를 포함해 출력하세요.

문의: "지난주에 결제한 연간 구독을 취소하고 싶습니다. 아직 거의 사용하지 않았습니다." """,
        "check": lambda s: "[환불]" in s,
        "desc": "분류와 간결한 출력",
    },
    {
        "id": "T4_규정답변",
        "prompt": """아래 규정에 근거해서만 답하세요. 근거 조항을 표기하세요.

[규정]
제7조 (연차 휴가의 이월) 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.

질문: 개인 사정으로 연차를 못 썼는데 내년으로 넘길 수 있나요?""",
        "check": lambda s: ("제7조" in s) and ("업무상" in s or "불가" in s or "없" in s),
        "desc": "근거 기반 답변과 조건 구분",
    },
    {
        "id": "T5_도구선택",
        "prompt": """사용 가능한 도구입니다.
- search_policy : 사내 규정을 검색합니다. 근무, 휴가, 보안 규칙 질문에 사용하세요.
- query_sales : 매출 데이터를 조회합니다. 지역별 월별 매출 질문에 사용하세요.
- calculate : 산술 계산을 합니다.

질문: "부산 3월 매출이 얼마야?"
호출할 도구 이름만 출력하세요.""",
        "check": lambda s: "query_sales" in s and "search_policy" not in s,
        "desc": "도구 선택 정확도",
    },
]

print(f"태스크 {len(tasks)}종 준비 완료")
for t in tasks:
    print(f"  {t['id']} : {t['desc']}")

태스크 5종 준비 완료
  T1_요약 : 정보 압축과 형식 준수
  T2_추출 : 구조화 출력
  T3_분류 : 분류와 간결한 출력
  T4_규정답변 : 근거 기반 답변과 조건 구분
  T5_도구선택 : 도구 선택 정확도


평가 세트는 **우리가 실제로 시킬 일과 같은 형태여야 합니다.** 다섯 태스크는 지금까지 배운 것과 대응되어 T2는 구조화 출력(에이전트의 도구 호출과 같은 능력), T4는 RAG 답변, T5는 도구 선택입니다. 벤치마크 점수가 아니라 우리가 쓸 능력을 재는 것입니다.

### Step 2.5 : 채점기 먼저 채점하기

모델을 올리기 전에 할 일이 하나 남았습니다. **채점기도 코드이고 코드에는 버그가 있습니다.**

이론 레슨 6.4절의 경고대로 측정 도구의 버그는 모델의 한계로 위장합니다. 채점기가 헐거우면 오답이 정답으로 세어지고 우리는 엉뚱한 모델을 고르고도 그 사실을 모릅니다.

방법도 이론 레슨 그대로 **일부러 틀린 답을 넣어 통과하는지 봅니다.** 정반대 답, 빈 문자열, 엉뚱한 문장을 채점기에 통과시켜 보고 통과되면 채점기가 잘못된 것입니다.

In [16]:
# 채점기 시험지. (답변, 통과해야 하는가) 쌍입니다.
# 맞는 답 하나와 **틀린 답 여러 개**를 넣는 것이 요령입니다
GRADER_TESTS = {
    "T1_요약": [
        ("신규 검색 기능 출시일을 10월 15일로 확정했습니다.\n"
         "레거시 API 폐기는 4분기로 연기했습니다.\n"
         "다음 회의는 10월 7일입니다.", True),
        ("출시일은 아직 정해지지 않았습니다.", False),
        ("회의 요약\n" + "\n".join(f"{i}. 항목" for i in range(1, 9)) + "\n10월 15일 확정", False),
        ("", False),
    ],
    "T2_추출": [
        ('{"고객사": "주식회사 한빛물산", "금액": 48000000, '
         '"납기": "2026-11-30", "담당자": "김서준"}', True),
        ('네, 정리해드리겠습니다. {"고객사": "주식회사 한빛물산", "금액": 4,800만원}', False),
        ("한빛물산과 4800만원 규모의 계약입니다.", False),
        ("", False),
    ],
    "T3_분류": [
        ("[환불]", True),
        ("[환불] 또는 [기타]로 분류할 수 있습니다.", False),
        ("환불 문의로 보입니다.", False),
        ("", False),
    ],
    "T4_규정답변": [
        ("제7조에 따라 개인 사정으로 사용하지 못한 연차는 다음 연도로 이월할 수 없습니다. "
         "다만 회사의 업무상 사유로 사용하지 못한 경우에는 인사팀장의 승인을 받아 "
         "최대 5일까지 이월할 수 있습니다.", True),
        ("제7조에 따르면 이월에 별도의 제한이 없습니다. 내년으로 넘기실 수 있습니다.", False),
        ("관련 규정이 없어 답변이 어렵습니다.", False),
        ("개인 사정으로는 이월할 수 없습니다.", False),
        # 규정 원문을 그대로 베껴 쓰고 결론은 반대인 답. 모델이 실제로 자주 내놓는 모양입니다
        ("[규정]\n제7조 (연차 휴가의 이월) 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 "
         "이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 "
         "인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.\n\n"
         "답변: 내년으로 넘길 수 있습니다.", False),
        ("", False),
    ],
    "T5_도구선택": [
        ("query_sales", True),
        ("query_sales 또는 search_policy를 쓰면 됩니다.", False),
        ("매출 관련 질문이므로 적절한 도구를 사용하세요.", False),
        ("", False),
    ],
}


def grade_the_graders(tasks, tests):
    """채점기를 채점합니다. 오작동 건수를 돌려줍니다"""
    bad = 0
    for t in tasks:
        for text, should in tests.get(t["id"], []):
            got = bool(t["check"](text))
            if got != should:
                bad += 1
                label = "오답을 통과시킴" if got else "정답을 떨어뜨림"
                print(f"  [{t['id']}] {label} : {(text[:38] or '(빈 문자열)')}")
    print(f"\n채점기 오작동 {bad}건" if bad else "\n채점기 검산 통과 : 오작동 0건")
    return bad


bad = grade_the_graders(tasks, GRADER_TESTS)

  [T2_추출] 오답을 통과시킴 : 네, 정리해드리겠습니다. {"고객사": "주식회사 한빛물산", "금액
  [T2_추출] 오답을 통과시킴 : 한빛물산과 4800만원 규모의 계약입니다.
  [T3_분류] 오답을 통과시킴 : [환불] 또는 [기타]로 분류할 수 있습니다.
  [T4_규정답변] 오답을 통과시킴 : 제7조에 따르면 이월에 별도의 제한이 없습니다. 내년으로 넘기실 수 
  [T4_규정답변] 오답을 통과시킴 : [규정]
제7조 (연차 휴가의 이월) 당해 연도에 사용하지 못한 연차

채점기 오작동 5건


가장 위험한 줄은 T4입니다. **"제한이 없습니다"라는 정반대 답이 통과했습니다.**

채점기에 들어 있는 `"없" in s`가 원인입니다. "이월할 수 없습니다"를 기대하고 넣은 조건입니다. 그런데 한국어에서 '없'은 아무 데나 들어가서 "규정이 없어"도 통과하고 "제한이 없습니다"도 통과합니다.

이 상태로 재면 규정을 정반대로 답하는 모델이 **정확도 100퍼센트**로 찍힙니다. 이론 레슨 6.4절에서 이 상태로 모델을 고르면 최악의 후보를 고를 수도 있다고 한 장면입니다.

T2와 T3에서도 같은 종류의 헐거움이 잡혔습니다. T2는 JSON이 깨져 있어도 글자만 들어 있으면 통과했고 T3는 라벨을 두 개 붙여도 통과했습니다.

T4에는 구멍이 하나 더 있습니다. **규정 원문을 그대로 베껴 쓴 뒤 결론을 반대로 적은 답도 통과합니다.** 베껴 쓴 규정 안에 '제7조', '이월할 수 없음', '업무상'이 다 들어 있기 때문입니다. 모델이 규정을 되풀이한 뒤 답을 붙이는 일은 실제로 자주 생깁니다.

In [17]:
# 채점기를 고칩니다. 원칙은 하나 : **판정기는 불합격을 기본값으로 둡니다**
# 통과할 조건이 명확히 확인될 때만 True를 돌려줍니다

LABELS = ["[기술지원]", "[영업]", "[환불]", "[기타]"]
DENY = ["이월할 수 없", "이월이 불가", "이월 불가", "넘길 수 없", "이월되지 않"]


def check_t2(s):
    """JSON으로 실제 파싱되는지까지 봅니다. 파싱이 안 되면 불합격"""
    m = re.search(r"\{.*\}", s, re.S)
    if not m:
        return False
    try:
        d = json.loads(m.group(0))
    except json.JSONDecodeError:
        return False
    if "한빛물산" not in str(d.get("고객사", "")):
        return False
    amount = str(d.get("금액", "")).replace(",", "").replace("원", "")
    return amount in ("48000000", "4800만", "4800")


def check_t3(s):
    """라벨이 정확히 하나여야 하고 그것이 [환불]이어야 합니다"""
    found = [x for x in LABELS if x in s]
    return found == ["[환불]"]


def check_t4(s):
    """근거 조항 + 결론 + 예외 조건, 셋을 모두 확인합니다.

    규정 원문을 베껴 쓴 줄은 모델의 결론으로 치지 않습니다. 규정은 "…한다", "…있다"로
    끝나는 문어체이고 답변은 "…습니다"로 끝나므로 줄 끝으로 가릅니다
    """
    if "제7조" not in s:                        # 근거 조항을 댔는가
        return False
    own = "\n".join(ln for ln in s.splitlines()
                    if not re.search(r"(한다|있다|없다)\.?\s*$", ln.strip()))   # 모델이 제 말로 쓴 줄만
    if not any(k in own for k in DENY):         # 개인 사정은 안 된다는 결론을 제 말로 냈는가
        return False
    return "업무상" in own                      # 예외 조건을 구분했는가


FIXED = {"T2_추출": check_t2, "T3_분류": check_t3, "T4_규정답변": check_t4}
for t in tasks:
    if t["id"] in FIXED:
        t["check"] = FIXED[t["id"]]

print("채점기 교체 완료. 다시 검산합니다\n")
bad = grade_the_graders(tasks, GRADER_TESTS)

채점기 교체 완료. 다시 검산합니다


채점기 검산 통과 : 오작동 0건


오작동이 0건이 되었습니다. 이제 이 채점기로 모델을 재도 됩니다.

채점기는 **통과 조건을 여러 개로 나누고 하나라도 안 맞으면 떨어뜨리도록** 고쳤습니다. T4는 근거 조항, 결론, 예외 조건 셋을 각각 확인하되 규정 원문을 베껴 쓴 줄(…한다, …있다로 끝나는 규정 어투)은 모델의 결론으로 치지 않습니다. 태스크가 근거 표기까지 요구했으므로 "개인 사정으로는 이월할 수 없습니다"처럼 결론만 맞고 조항을 안 댄 답도 떨어집니다.

앞으로 평가 세트를 만들 때마다 이 절을 먼저 하십시오. **모델을 올리는 데는 10분이 걸리고 채점기를 검산하는 데는 10초가 걸립니다.**

## Step 3 : 모델별 측정

모델을 하나씩 올리고 내리며 잽니다.

**내리는 코드**를 조심하십시오. `del`은 이름을 지울 뿐 객체를 지우지 않습니다. 흔히 쓰는 `def unload(tok, model): del model, tok`는 함수 안의 이름만 지우고 부르는 쪽의 `tok`과 `model`이 여전히 모델을 붙잡고 있어 **한 바이트도 해제하지 않습니다.** 오류도 나지 않아서 다음 모델을 올릴 때 메모리 부족으로 멈추고 나서야 알게 됩니다.

그래서 모델과 토크나이저를 **딕셔너리 한 곳에만** 담아 참조를 흩어 놓지 않고 내릴 때 **해제 전후의 VRAM을 찍어** 실제로 줄었는지 눈으로 확인합니다.

In [18]:
SYSTEM = "당신은 한국 회사의 업무 보조입니다. 답변은 반드시 한국어로 작성하세요."


def vram_gb():
    """지금 텐서가 잡고 있는 VRAM(GB)"""
    return torch.cuda.memory_allocated() / 1e9


def load_model(name):
    """모델을 올려 딕셔너리에 담아 돌려줍니다.

    참조를 한 곳에만 두는 것이 뒤의 unload를 위해 중요합니다
    """
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb,
                                                 device_map="auto")
    model.eval()
    return {"repo": name, "tok": tok, "model": model}


def unload(h):
    """모델을 내리고 **실제로 줄었는지** 숫자로 보여줍니다.

    del만으로는 부족합니다. 세 가지를 순서대로 해야 합니다.
      1) 참조를 끊는다 (h 안의 값을 None으로). h가 유일한 참조여야 효과가 있습니다
      2) gc.collect()  : 파이썬이 객체를 실제로 회수하게 합니다
      3) empty_cache() : 회수된 자리를 드라이버에 돌려줍니다
    """
    before = vram_gb()
    h["tok"] = h["model"] = None
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    after = vram_gb()
    print(f"  [해제] VRAM {before:.2f} -> {after:.2f} GB  (줄어든 양 {before - after:.2f})")
    if before > 0.1 and after > before * 0.3:
        print("  ! 거의 줄지 않았습니다. 다른 변수가 모델을 붙잡고 있는지 확인하세요")
    return after


def build_prompt(tok, user):
    """시스템 메시지로 출력 언어를 고정합니다.

    시스템 역할을 지원하지 않는 채팅 템플릿(gemma 계열)이 있습니다.
    그때는 예외를 올리지 않고 사용자 메시지 앞에 붙이는 쪽으로 넘어갑니다
    """
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    try:
        return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    except Exception:
        merged = [{"role": "user", "content": SYSTEM + "\n\n" + user}]
        return tok.apply_chat_template(merged, add_generation_prompt=True, tokenize=False)


def generate(tok, model, prompt, max_new_tokens=300):
    text = build_prompt(tok, prompt)
    ids = tok(text, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    elapsed = time.time() - t0
    n_new = out.shape[1] - ids["input_ids"].shape[1]
    ans = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return ans, n_new / elapsed, elapsed

In [19]:
# 모델별로 5개 태스크를 수행하고 정확도, 처리 속도, VRAM을 측정합니다
TEST_MODELS = ["Qwen/Qwen2.5-3B-Instruct", "meta-llama/Llama-3.2-3B-Instruct"]

results = {}
for name in TEST_MODELS:
    print(f"\n=== {name} ===")
    repo = models[name]["repo"]            # 게이트가 걸렸으면 미러 이름이 들어 있습니다
    if repo is None:                       # 원본도 미러도 막힌 경우
        results[name] = {"skipped": models[name]["gate"]}
        print(f"  건너뜀 : {models[name]['gate']}")
        continue
    if repo != name:
        print(f"  (미러 사용 : {repo})")

    try:
        h = load_model(repo)
    except Exception as e:
        results[name] = {"skipped": f"로드 실패 : {type(e).__name__}"}
        print(f"  건너뜀 : 로드 실패 {type(e).__name__}")
        continue

    vram = vram_gb()
    per_task, speeds, answers = {}, [], {}

    for t in tasks:
        ans, tps, _ = generate(h["tok"], h["model"], t["prompt"])
        ok = bool(t["check"](ans))
        per_task[t["id"]] = ok
        speeds.append(tps)
        answers[t["id"]] = ans.strip()
        print(f"  [{'O' if ok else 'X'}] {t['id']} ({tps:.1f} tok/s)")

    results[name] = {
        "repo": repo,
        "gate": models[name]["gate"],
        "accuracy": sum(per_task.values()) / len(per_task),
        "per_task": per_task,
        "tps": float(np.mean(speeds)),
        "vram": vram,
        "answers": answers,
    }
    unload(h)


=== Qwen/Qwen2.5-3B-Instruct ===


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

  [O] T1_요약 (15.8 tok/s)
  [X] T2_추출 (15.8 tok/s)
  [X] T3_분류 (13.3 tok/s)
  [X] T4_규정답변 (15.9 tok/s)
  [X] T5_도구선택 (12.5 tok/s)
  [해제] VRAM 2.25 -> 0.01 GB  (줄어든 양 2.24)


In [20]:
# 비교표. 태스크별 칸을 남깁니다. 평균 한 줄은 정보를 버립니다 (이론 레슨 3.6절)
cols = [t["id"].split("_")[0] for t in tasks]
print(f"{'모델':<22} " + " ".join(f"{c:>4}" for c in cols)
      + f" {'정확도':>7} {'tok/s':>7} {'VRAM':>7} {'게이트':>14}")
print("-" * 96)
for name, r in results.items():
    short = name.split("/")[-1][:22]
    if "skipped" in r:
        print(f"{short:<22} {'건너뜀 : ' + r['skipped']:>60}")
        continue
    marks = " ".join(f"{'O' if r['per_task'][t['id']] else 'X':>4}" for t in tasks)
    print(f"{short:<22} {marks} {r['accuracy']:>6.0%} {r['tps']:>7.1f} "
          f"{r['vram']:>6.2f}G {r['gate']:>14}")

모델                       T1   T2   T3   T4   T5     정확도   tok/s    VRAM            게이트
------------------------------------------------------------------------------------------------
Qwen2.5-3B-Instruct       O    O    O    X    O    80%    11.4   2.06G             열림
Llama-3.2-3B-Instruct     O    X    X    X    X    20%    14.7   2.25G      게이트 -> 미러


**표에서 볼 것**

**첫째, 태스크별 칸입니다.** 정확도 평균만 있으면 60퍼센트와 60퍼센트가 같아 보이지만 칸을 나누면 한쪽은 구조화 출력에서, 다른 쪽은 분류와 도구 선택에서 틀렸다는 것이 보입니다. 보완 방법이 완전히 다릅니다.

**둘째, 게이트 칸입니다.** 사내에 옮길 때 승인 절차가 일정에 들어가야 하므로 미러로 받은 모델은 그 사실이 표에 남아 있어야 합니다.

숫자만 보지 말고 **틀린 답변을 직접 읽어 보십시오.** 왜 틀렸는지가 모델 선택의 실질적 근거입니다. 같은 정확도 40퍼센트라도 형식만 틀린 경우는 컨텍스트와 하네스 노드에서 배운 방법으로 고칠 수 있지만 내용이 완전히 엉뚱한 경우는 고치기 어렵습니다.

In [21]:
# 틀린 답변을 확인합니다
for name, r in results.items():
    if "skipped" in r:
        continue
    print(f"=== {name} ===")
    for t in tasks:
        if not r["per_task"][t["id"]]:
            print(f"  [{t['id']}] {r['answers'][t['id']][:150]}")
    print()

=== Qwen/Qwen2.5-3B-Instruct ===
  [T4_규정답변] 물론입니다. 규정에 따르면 개인 사정으로 연차를 사용하지 못한 경우, 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 연차 휴가를 이월할 수 있습니다. 따라서 개인 사정으로 연차를 못 쓰셨다면 인사팀장의 승인이 필요합니다. 이를 받으시면 내년으로 연차 휴가

=== meta-llama/Llama-3.2-3B-Instruct ===
  [T2_추출] {"고객사": "주식회사 한빛물산", "금액": 4800000, "납기": "2026-11-30", "담당자": "영업2팀 김서준 과장"}
  [T3_분류] [기술지원]
  [T4_규정답변] 당신의 경우, 개인 사정으로 연차를 못 썼습니다. 따라서, 이월이 원칙으로 되어 있기 때문에, 인사팀장의 승인을 받아서 다음 연도 1분기 내로 이월하여 사용할 수 있습니다.
  [T5_도구선택] search_policy



## Step 4 : 블라인드 정성 평가

정량 지표가 못 잡는 것이 있습니다. 답변의 자연스러움, 한국어 품질, 불필요한 사족입니다.

이름을 알면 편향이 생기므로 모델 이름을 가리고 답변만 보고 평가합니다.

In [22]:
import random

# 답변을 섞어서 출력합니다. 어느 모델인지 모르는 상태로 평가하세요
pairs = []
for t in tasks:
    for name, r in results.items():
        if "skipped" in r:
            continue
        pairs.append({"task": t["id"], "model": name, "answer": r["answers"][t["id"]]})

random.seed(7)
random.shuffle(pairs)

for i, p in enumerate(pairs):
    print(f"--- 샘플 {i} ({p['task']}) ---")
    print(p["answer"][:300])
    print()

--- 샘플 0 (T5_도구선택) ---
query_sales

--- 샘플 1 (T2_추출) ---
{"고객사": "주식회사 한빛물산", "금액": 4800000, "납기": "2026-11-30", "담당자": "영업2팀 김서준 과장"}

--- 샘플 2 (T1_요약) ---
9월 정기 개발회의 결과 요약:

1. 신규 검색 기능 출시일은 10월 15일로 확정되었으며, QA 기간은 2주 동안 확보되었습니다.
2. 레거시 API 폐기는 4분기으로 연기되었으며, 외부 연동 고객 3곳의 마이그레이션은 아직 완료되지 않았습니다.
3. 인프라 비용 절감을 위해 GPU 인스턴스 사용률을 40퍼센트로 낮추고 검토할 예정입니다.
4. 다음 회의는 10월 7일로 정해졌습니다.

--- 샘플 3 (T3_분류) ---
[환불]

--- 샘플 4 (T4_규정답변) ---
당신의 경우, 개인 사정으로 연차를 못 썼습니다. 따라서, 이월이 원칙으로 되어 있기 때문에, 인사팀장의 승인을 받아서 다음 연도 1분기 내로 이월하여 사용할 수 있습니다.

--- 샘플 5 (T1_요약) ---
9월 정기 개발회의에서 신규 검색 기능 출시 일정이 10월 15일로 확정되었고, QA 기간 2주를 확보하기로 했습니다. 또한 레거시 API 폐기는 4분기로 연기되었으며, 외부 연동 고객 3곳의 마이그레이션 작업이 완료되지 않았습니다. 인프라 비용 절감안에 대해 논의하였고, GPU 인스턴스 사용률을 40%로 낮추는 방안으로 인프라 축소 검토가 이루어졌습니다. 다음 회의는 10월 7일로 예정되었습니다.

--- 샘플 6 (T5_도구선택) ---
search_policy

--- 샘플 7 (T4_규정답변) ---
물론입니다. 규정에 따르면 개인 사정으로 연차를 사용하지 못한 경우, 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 연차 휴가를 이월할 수 있습니다. 따라서 개인 사정으로 연차를 못 쓰셨다면 인사팀장의 승인이 필요합니다. 이를 받으시면 내년으로 연차 휴가를 이월할 수 있습니다. 제7조

In [23]:
# 평가표를 채우세요. 5점 척도로 각 샘플을 평가합니다
# 기준 : 한국어 자연스러움, 지시 준수, 사족 없음

my_scores = {
    0: 5,  # T5: 도구 선택 완벽
    1: 3,  # T2: JSON 좋으나 금액 0 누락 (480만원)
    2: 3,  # T1: 3줄 제한 어김 (5줄 작성)
    3: 5,  # T3: [환불] 정확한 분류
    4: 1,  # T4: 사규 정반대 왜곡 (심각한 환각)
    5: 4,  # T1: 요약 조건 준수
    6: 1,  # T5: 엉뚱한 도구 선택 (search_policy)
    7: 2,  # T4: 조항은 적었으나 개인 사정 이월 왜곡
    8: 5,  # T2: 금액 48000000 완벽 추출
    9: 1,  # T3: 환불 문의인데 [기술지원]으로 오분류
}

# 채점 후 실행하면 모델별 평균이 나옵니다
if my_scores:
    agg = {}
    for i, s in my_scores.items():
        m = pairs[i]["model"]
        agg.setdefault(m, []).append(s)
    for m, v in agg.items():
        print(f"{m.split('/')[-1]:<30} 정성 평균 {np.mean(v):.2f}")
else:
    print("my_scores를 채운 뒤 실행하세요")

Qwen2.5-3B-Instruct            정성 평균 4.20
Llama-3.2-3B-Instruct          정성 평균 1.80


## Step 5 : 라이선스 체크리스트 작성

이론 레슨 4.1절에서 라이선스가 첫 번째 필터라고 했습니다. 후보 모델의 라이선스를 직접 확인해 표를 채웁니다.

각 모델의 Hugging Face 카드에서 라이선스 섹션을 읽고 카드에 링크된 원문 라이선스까지 확인하십시오. 게이트 여부는 Step 1에서 이미 확인했으므로 손으로 다시 적지 않고 **읽어 옵니다.**

In [24]:
# 실습 : 후보 모델의 라이선스를 조사해 아래 표를 채우세요
# 조사는 **원본 저장소 기준**입니다. 미러로 받았더라도 라이선스는 원본을 따릅니다
license_check = [
    {
        "model": "Qwen/Qwen2.5-7B-Instruct",
        "license_name": "Apache 2.0",
        "commercial": True,
        "user_limit": "없음",
        "derivative_ok": True,
        "output_reuse": "가능",
        "attribution": "필요 (저작권 고지)",
        "notes": "가장 자유로운 표준 오픈소스 라이선스",
    },
    {
        "model": "meta-llama/Llama-3.2-3B-Instruct",
        "license_name": "Llama 3.2 Community",
        "commercial": True,
        "user_limit": "MAU 7억 명",
        "derivative_ok": True,
        "output_reuse": "불가 (타 모델 학습 금지)",
        "attribution": "필요 (Built with Llama)",
        "notes": "자회사 합산 사용자 수 확인 필요, 파생 모델명 규칙 준수",
    },
    {
        "model": "google/gemma-2-2b-it",
        "license_name": "Gemma Terms of Use",
        "commercial": True,
        "user_limit": "없음",
        "derivative_ok": True,
        "output_reuse": "가능",
        "attribution": "필요",
        "notes": "구글 독자 약관, 금지 사용 정책(AUP) 준수",
    },
]

# 확인 날짜를 함께 남깁니다. 라이선스는 바뀝니다 (이론 레슨 4.5절)
CHECKED_ON = time.strftime("%Y-%m-%d")

for c in license_check:
    gate = models.get(c["model"], {}).get("gate", "확인 안 함")
    print(f"{c['model']}  (확인일 {CHECKED_ON})")
    print(f"  라이선스     : {c['license_name'] or '(조사 필요)'}")
    print(f"  상업적 사용  : {c['commercial']}")
    print(f"  파생 모델    : {c['derivative_ok']}")
    print(f"  게이트       : {gate}")
    print()

Qwen/Qwen2.5-7B-Instruct  (확인일 2026-10-08)
  라이선스     : Apache 2.0
  상업적 사용  : True
  파생 모델    : True
  게이트       : 열림

meta-llama/Llama-3.2-3B-Instruct  (확인일 2026-10-08)
  라이선스     : Llama 3.2 Community
  상업적 사용  : True
  파생 모델    : True
  게이트       : 게이트 -> 미러

google/gemma-2-2b-it  (확인일 2026-10-08)
  라이선스     : Gemma Terms of Use
  상업적 사용  : True
  파생 모델    : True
  게이트       : 게이트 -> 미러



**모델 카드의 라이선스 표시와 실제 라이선스 문서의 조건이 다를 수 있습니다.** 카드에는 짧게 적혀 있어도 원문에 조건이 붙어 있기도 합니다.

게이트 칸은 라이선스 조건이 아니라 **일정 항목**입니다. 승인에 며칠이 걸릴 수 있고 새 팀원이 합류하면 각자 다시 받아야 하고 폐쇄망에서는 미리 받아 두지 않으면 아예 못 받습니다. 🖥 **로컬 작업** : 사내 구축 계획서에는 이 줄이 일정표의 한 항목으로 들어가야 합니다.

조사 결과를 그대로 믿지 마십시오. 이 작업의 목적은 **법무팀에 전달할 확인 항목을 정리하는 것**이고 최종 판단은 법무 검토입니다.

## Step 6 : 다음 노드 프로젝트 설계서 작성

다음 노드에서 진행할 사내 데이터 파인튜닝 프로젝트의 설계서를 지금 씁니다.

**이론 레슨 6.2절 첫머리의 7단계 표**를 그대로 채웁니다. 아래 딕셔너리의 일곱 칸이 그 표의 일곱 줄입니다. 특히 3단계 프라이버시 경계를 4단계 기술 방식보다 먼저 정하는 순서를 지키십시오. 오늘 배운 모델 선택은 그중 5단계입니다.

In [25]:
# 실습 : 설계서를 작성하세요. 각 항목을 구체적으로 채웁니다
# 일곱 칸은 이론 레슨 6.2절의 7단계 표와 같은 순서입니다

design = {
    "1_요구사항": {
        "자동화_대상": "사내 복무규정, 인사제도, 복리후생, 업무 가이드 관련 반복 질의응답",
        "사용자": "사내 전 임직원 (약 500명, 피크타임 동시 접속 5~10명 예상)",
        "목표_응답시간": "첫 토큰 생성(TTFT) 1.5초 이내, 전체 응답 4초 이내",
        "목표_정확도": "규정 근거 답변 정확도 90% 이상 (환각 및 허위 사규 안내 0건)",
        "성공_기준": "사내 대표 질의 50건 블라인드 테스트에서 정답률 90% 달성 및 인사팀 반복 문의 40% 감소",
    },
    "2_데이터": {
        "사용_데이터": "취업규칙, 인사규정, 연차/휴가 매뉴얼, 경비처리 지침서 등 사내 공식 문서 (PDF, DOCX)",
        "데이터_위치": "사내 인트라넷 보안 파일 서버 (폐쇄망 내부 스토리지)",
        "민감도": "사내 대외비 (2급 비밀/영업비밀 제외, 임직원 개인정보 및 사내 내부 규정 포함)",
        "규모": "약 80건 문서, 텍스트 기준 약 40만 토큰 (청크 약 1,500개)",
    },
    "3_프라이버시_경계": {
        "내부_영역": "임베딩 연산, 벡터 DB 검색, LLM 추론(vLLM), 사용자 질문 로그 전체를 사내 온프레미스망 내에서만 처리",
        "외부_허용": "없음 (공용 퍼블릭 클라우드 및 외부 API 전송 전면 차단)",
        "근거_규정": "사내 정보보안 기본지침 제15조 (사내 업무 데이터의 외부 AI 서비스 입력 금지)",
    },
    "4_기술_방식": {
        "선택": "RAG",
        "선택_이유": "사규 개정 시 즉시 문서만 교체하여 반영 가능하며, 모델 가중치에 기밀이 영구 각인되는 암기(Memorization) 위험을 원천 차단하기 위함",
        "먼저_시도할_것": "RAG (사내 문서 기반 검색 증강을 먼저 검증하고, 파인튜닝은 말투/포맷 고정이 필요할 때만 후순위 검토)",
    },
    "5_모델과_스택": {
        "선정_모델": "Qwen/Qwen2.5-3B-Instruct",
        "선정_근거_라이선스": "Apache 2.0으로 상업적 사용, 수정, 사내 배포에 제약 및 특허 리스크가 전혀 없음",
        "선정_근거_VRAM": "Step 1 계산 결과: 4bit 양자화 가중치 2.01GB + KV 캐시 및 오버헤드 포함 약 2.8GB로, 보유 중인 소형 GPU(12GB~24GB) 1장으로 여유 있게 서빙 가능",
        "선정_근거_평가": "Step 3/4 측정 결과: 사내 업무 5종 블라인드 평가에서 Llama 3.2 대비 한국어 표현력, JSON 추출, 조항 근거 답변 품질이 월등함",
        "게이트_여부": "열림 (별도의 접근 승인 대기나 라이선스 동의 절차 없이 즉시 배포 가능하여 개발 일정 단축)",
        "임베딩_모델": "BAAI/bge-m3",
        "벡터DB": "Chroma (초기 온프레미스 구축 및 경량 운영에 적합)",
    },
    "6_아키텍처": {
        "구성_요소": [
            "사내 메신저(Slack/Teams) UI",
            "FastAPI 사내 중계 서버 (인증 및 PII 마스킹 전처리)",
            "Chroma Vector DB (사내 문서 청크 적재)",
            "vLLM 추론 엔진 (Qwen2.5-3B-Instruct AWQ 구동)",
        ],
        "데이터_흐름": "직원 질문 입력 -> 사내 중계 서버(개인정보 마스킹) -> Chroma DB 관련 사규 검색 -> 프롬프트 조립 -> vLLM(Qwen 3B) 답변 생성 -> 사용자 반환",
        "경계_표시": "사내 방화벽 및 망분리 영역 안쪽에 모든 컴포넌트(DB, LLM, API)를 배치하고 외부 인터넷 통신 포트는 전면 인바운드/아웃바운드 차단",
    },
    "7_운영": {
        "모니터링_지표": "GPU VRAM 사용량, 초당 생성 토큰 수(TPS), TTFT(초기 응답 지연), 사용자 피드백(좋아요/싫어요 비율)",
        "업데이트_주기": "사규 및 지침 개정 시 즉시 문서 재임베딩 (정기 주기는 매월 1회 배치 동기화)",
        "장애_대응": "LLM 서버 장애 시 '인사팀 대표 문의 채널(내선 번호)' 안내 메시지로 즉시 대체(Fallback)",
        "권한_관리": "사내 사번 기반 SSO 연동 및 부서별/직급별 열람 불가 문서(예: 임원 전용 지침) 검색 필터링(RBAC) 적용",
        "재검토_조건": "동시 접속자가 30명 이상으로 급증해 GPU 병목이 발생하거나, 더 우수한 한국어 경량 오픈소스 모델(예: 차세대 7B급 이상)이 아파치 2.0으로 출시될 경우",
    },
}

# 작성 후 저장합니다
with open("design_doc.json", "w", encoding="utf-8") as f:
    json.dump(design, f, ensure_ascii=False, indent=2)
print("design_doc.json 저장 완료")

design_doc.json 저장 완료


In [26]:
# 작성한 설계서의 완성도를 자체 점검합니다
def review_design(d):
    missing = []
    for section, items in d.items():
        for k, v in items.items():
            if not v or (isinstance(v, list) and not v):
                missing.append(f"{section}.{k}")
    return missing


missing = review_design(design)
if missing:
    print(f"미작성 항목 {len(missing)}개")
    for m in missing[:15]:
        print(f"  - {m}")
else:
    print("모든 항목 작성 완료")

모든 항목 작성 완료


## Step 7 : 선택 심화 트랙

다섯 트랙 중 **두 개를 선택**해 수행하고 공유합니다.

| 트랙 | 주제 | 성격 | 난이도 |
| :---: | :--- | :---: | :---: |
| **1** | **첫 토큰 지연과 처리량 분리 측정** | 측정 | 하 |
| **2** | **작은 모델 + 하네스 vs 큰 모델 단독** | 측정 | 중 |
| **3** | **컨텍스트 길이별 VRAM 실측과 추정 검증** | 측정 | 중 |
| **4** | **태스크별 모델 분담 설계** | 설계 | 중 |
| **5** | **양자화 정밀도와 품질 저하 측정** | 측정 | 상 |

#### 트랙 1 : 첫 토큰 지연과 처리량

사용자 체감 속도는 **첫 토큰까지의 지연**(TTFT)과 **이후 토큰 생성 속도**(TPS) 두 지표로 나뉩니다. 챗봇에서는 TTFT가 체감에 크게 영향을 주고 긴 문서 생성에서는 TPS가 중요합니다.

In [27]:
import time
import torch
import numpy as np


def measure_latency(tok, model, prompt, max_new_tokens=200):
    text = build_prompt(tok, prompt)
    ids = tok(text, return_tensors="pt").to(model.device)

    # 1. TTFT 측정 (첫 토큰 1개만 생성)
    torch.cuda.synchronize()  # GPU 시작 대기
    t0 = time.time()
    with torch.no_grad():
        model.generate(
            **ids,
            max_new_tokens=1,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    torch.cuda.synchronize()  # GPU 연산 완료 대기
    ttft = time.time() - t0

    # 2. 전체 생성 시간 측정
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    torch.cuda.synchronize()
    total = time.time() - t0

    # 3. 토큰 개수 및 TPS(초당 생성 토큰 수) 계산
    n_new = out.shape[1] - ids["input_ids"].shape[1]
    # 첫 토큰은 TTFT에서 빠졌으므로 (n_new - 1) 개를 (total - ttft) 시간으로 나눔
    generation_time = max(total - ttft, 1e-6)
    tps = (n_new - 1) / generation_time if n_new > 1 else 0.0

    return {
        "prompt_tokens": ids["input_ids"].shape[1],
        "ttft": ttft,
        "tps": tps,
        "total": total,
    }


# ==========================================
# 실행부 (워밍업 추가)
# ==========================================
h = load_model(models["Qwen/Qwen2.5-3B-Instruct"]["repo"])

# 🔥 [개선점 1] 워밍업(Warm-up): 잠든 GPU 깨우기
print("GPU 워밍업 진행 중...")
_ = measure_latency(h["tok"], h["model"], "안녕하세요", max_new_tokens=5)
print("워밍업 완료! 본 측정 시작\n")

# 본 측정
for n in [50, 500, 2000]:
    p = "다음 문서를 요약하세요.\n\n" + (
        "사내 규정 조항 예시 문장입니다. " * (n // 10)
    )
    r = measure_latency(h["tok"], h["model"], p, max_new_tokens=100)
    print(
        f"입력 {r['prompt_tokens']:>5} 토큰 : TTFT {r['ttft']:.3f}초 | 순수 생성"
        f" TPS {r['tps']:>5.1f} tok/s"
    )

_ = unload(h)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GPU 워밍업 진행 중...
워밍업 완료! 본 측정 시작

입력   100 토큰 : TTFT 0.111초 | 순수 생성 TPS  12.5 tok/s
입력   595 토큰 : TTFT 0.318초 | 순수 생성 TPS  12.7 tok/s
입력  2245 토큰 : TTFT 1.685초 | 순수 생성 TPS  12.8 tok/s
  [해제] VRAM 2.07 -> 0.01 GB  (줄어든 양 2.07)


<details>
<summary>정답 보기</summary>

```python
def measure_latency(tok, model, prompt, max_new_tokens=200):
    text = build_prompt(tok, prompt)
    ids = tok(text, return_tensors="pt").to(model.device)

    # TTFT : 첫 토큰 하나만 생성
    t0 = time.time()
    with torch.no_grad():
        model.generate(**ids, max_new_tokens=1, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    ttft = time.time() - t0

    # 전체 생성
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    total = time.time() - t0
    n_new = out.shape[1] - ids["input_ids"].shape[1]

    tps = (n_new - 1) / max(total - ttft, 1e-6)
    return {"prompt_tokens": ids["input_ids"].shape[1], "ttft": ttft,
            "tps": tps, "total": total}

# 프롬프트 길이를 바꿔가며 TTFT가 어떻게 변하는지 봅니다
h = load_model(models["Qwen/Qwen2.5-3B-Instruct"]["repo"])
for n in [50, 500, 2000]:
    p = "다음 문서를 요약하세요.\n\n" + ("사내 규정 조항 예시 문장입니다. " * (n // 10))
    r = measure_latency(h["tok"], h["model"], p, max_new_tokens=100)
    print(f"입력 {r['prompt_tokens']:>5} 토큰 : TTFT {r['ttft']:.2f}초, TPS {r['tps']:.1f}")
_ = unload(h)
```

프롬프트 전체를 한 번 처리해야 첫 토큰이 나오므로 **입력이 길어지면 TTFT가 늘어납니다.** TPS는 상대적으로 일정합니다.

RAG에서 검색 문서를 많이 넣으면 정확도는 올라갈 수 있지만 TTFT가 늘어 사용자가 답답해집니다. 컨텍스트와 하네스 노드의 top-k 조정은 정확도만의 문제가 아니었습니다.

**실행 순서** : 실습 셀 아래에 이 블록을 덧붙여 실행하십시오. 모델을 한 번 올리므로 2~3분이 걸리고 마지막 `unload`의 해제 로그까지 확인한 뒤 다른 트랙으로 넘어가십시오.

</details>

#### 트랙 2 : 작은 모델 + 하네스 vs 큰 모델 단독

컨텍스트와 하네스 노드에서 인용한 주장, "괜찮은 모델에 훌륭한 하네스를 붙인 쪽이, 훌륭한 모델에 나쁜 하네스를 붙인 쪽을 이깁니다"를 검증합니다.

In [ ]:
# 실습 : 두 조건을 비교하세요
#  조건 A : 큰 모델(7B) + 하네스 없음
#  조건 B : 작은 모델(3B) + 가이드 + 센서 + 재시도 루프
# 태스크는 Step 2의 5종을 씁니다. 고쳐 둔 check 함수가 센서 역할을 합니다

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

A. 7B 하네스 없음 : 5/5
  [해제] VRAM 5.57 -> 0.01 GB  (줄어든 양 5.56)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

B. 3B + 하네스 : 4/5 (평균 시도 1.4회)
  [해제] VRAM 2.07 -> 0.01 GB  (줄어든 양 2.07)

[B의 T4_규정답변 답] [규정]
제7조 (연차 휴가의 이월) 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.

질문: 개인 사정으로 연차를 못 썼는데 내년으로 넘길 수 있나요?

답변: 개인 사정으로 연차를 못 썼는데, 내년으로 넘길 수 있습니다. 하지만 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있습니다.

태스크              A(7B)      B(3B+하네스)
--------------------------------------
T1_요약                O              O
T2_추출                O              O
T3_분류                O              O
T4_규정답변              O              X
T5_도구선택              O              O

<details>
<summary>정답 보기</summary>

```python
def run_with_harness(tok, model, task, max_retry=3):
    """가이드를 추가하고, check 실패 시 피드백과 함께 재시도"""
    guide = "\n\n[작성 규칙]\n지시된 출력 형식을 정확히 지키세요. 설명이나 사족을 붙이지 마세요."
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": task["prompt"] + guide}]
    ans = ""

    for attempt in range(max_retry):
        try:
            text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        except Exception:                      # 시스템 역할을 못 받는 템플릿
            merged = [{"role": "user", "content": SYSTEM + "\n\n" + m["content"]}
                      if m["role"] == "system" else m for m in msgs]
            text = tok.apply_chat_template(merged, add_generation_prompt=True, tokenize=False)
        ids = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=300,
                                 do_sample=False, pad_token_id=tok.eos_token_id)
        ans = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

        if task["check"](ans):
            return ans, True, attempt + 1

        msgs.append({"role": "assistant", "content": ans})
        msgs.append({"role": "user",
                     "content": "출력이 요구된 형식이나 조건을 만족하지 않습니다. "
                                "지시를 다시 읽고 형식을 정확히 지켜 다시 작성하세요."})
    return ans, False, max_retry

# 조건 A : 7B 단독
h = load_model(models["Qwen/Qwen2.5-7B-Instruct"]["repo"])
a_hits, a_tasks = 0, {}
for t in tasks:
    ans, _, _ = generate(h["tok"], h["model"], t["prompt"])
    ok = bool(t["check"](ans))
    a_tasks[t["id"]] = ok
    a_hits += ok
print(f"A. 7B 하네스 없음 : {a_hits}/{len(tasks)}")
unload(h)

# 조건 B : 3B + 하네스
h = load_model(models["Qwen/Qwen2.5-3B-Instruct"]["repo"])
b_hits, b_tries, b_tasks, b_ans = 0, 0, {}, {}
for t in tasks:
    ans, ok, tries = run_with_harness(h["tok"], h["model"], t)
    b_tasks[t["id"]] = ok
    b_ans[t["id"]] = ans
    b_hits += ok
    b_tries += tries
print(f"B. 3B + 하네스 : {b_hits}/{len(tasks)} (평균 시도 {b_tries/len(tasks):.1f}회)")
unload(h)

# Step 3에서 3B가 틀렸던 태스크의 답이 하네스를 붙이자 어떻게 달라졌는지 읽어 봅니다
step3 = results["Qwen/Qwen2.5-3B-Instruct"]["per_task"]
for t in tasks:
    if not step3[t["id"]]:
        print(f"\n[B의 {t['id']} 답] {b_ans[t['id']][:600]}")

# 어느 태스크에서 갈렸는지를 봅니다
print(f"\n{'태스크':<14} {'A(7B)':>7} {'B(3B+하네스)':>14}")
print("-" * 38)
for t in tasks:
    print(f"{t['id']:<14} {'O' if a_tasks[t['id']] else 'X':>7} "
          f"{'O' if b_tasks[t['id']] else 'X':>14}")
```

이번 실행에서 A(7B 단독)는 5/5, B(3B + 하네스)는 4/5이고 B의 평균 시도는 1.4회입니다. 네 태스크는 첫 시도에 통과했고 **T4만 세 번(최대 재시도)을 다시 써도 떨어졌습니다.** 늘어난 0.4회가 하네스의 비용(지연과 토큰)이고 그 비용을 내고도 T4는 고쳐지지 않았습니다.

위에 찍힌 B의 T4 답을 읽어 보십시오. 규정과 질문을 그대로 되풀이한 뒤 '내년으로 넘길 수 있습니다'라고 반대로 답했습니다. 조항 번호와 예외 조건은 적었으니 형식은 지켰고 내용이 틀렸습니다. '형식이나 조건을 만족하지 않습니다'라는 피드백을 두 번 더 받아도 결론은 바뀌지 않았습니다. 이론 레슨 6.3절의 "형식 문제는 하네스로 고쳐지지만 내용 문제는 고치기 어렵습니다"를 숫자로 확인한 것입니다. 트랙 5를 했다면 알 수 있듯 이 T4를 고치는 것은 하네스가 아니라 정밀도(fp16)였습니다.

재시도 루프가 도는 모습을 처음부터 보려면 `guide`를 빈 문자열로 바꿔 다시 돌려 보십시오. Step 3과 같은 첫 답이 나오고 피드백 뒤의 두 번째 답이 어떻게 달라지는지 볼 수 있습니다.

응답 시간이 중요한 서비스면 7B를 쓰고 배치 처리라면 3B에 하네스를 붙이는 편이 자원 효율이 좋습니다. 이번 결과에서 3B는 VRAM을 5.57GB가 아니라 2.07GB만 쓰면서 형식 태스크 네 개는 같은 점수를 냈고 규정 해석 하나를 놓쳤습니다. 그 하나가 우리 업무의 핵심이라면 7B나 fp16 3B로 가고, 아니라면 3B + 하네스가 답입니다.

**실행 순서** : 이 블록을 실습 셀 아래에 덧붙여 한 번에 실행하십시오. 모델을 두 번 올리므로 10~15분이 걸립니다. 중간에 `[해제]` 로그가 두 번 찍히는지 확인하십시오. 찍히지 않으면 두 번째 모델을 올릴 때 메모리 부족으로 멈춥니다.

</details>

#### 트랙 3 : VRAM 추정 검증

Step 1의 계산기가 실제와 얼마나 맞는지, 추정을 믿을 수 있는지 확인합니다.

<details>
<summary>정답 보기</summary>

```python
h = load_model(models["Qwen/Qwen2.5-3B-Instruct"]["repo"])
cfg = configs["Qwen/Qwen2.5-3B-Instruct"]
limit = max_ctx_of(cfg) or 4096

print(f"{'시퀀스':>8} {'추정(GB)':>10} {'실측(GB)':>10} {'오차':>8}")
print("-" * 40)
for seq in [512, 1024, 2048, 4096]:
    if seq > limit:                       # 모델이 받을 수 있는 길이를 넘지 않게
        print(f"{seq:>8} {'모델 최대 길이 초과로 건너뜀':>30}")
        continue
    filler = "사내 규정 문서 예시 문장입니다. " * (seq // 8)
    ids = h["tok"](filler, return_tensors="pt", truncation=True,
                   max_length=seq).to(h["model"].device)

    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        h["model"].generate(**ids, max_new_tokens=32, do_sample=False,
                            pad_token_id=h["tok"].eos_token_id)
    actual = torch.cuda.max_memory_allocated() / 1e9
    est = estimate_vram(cfg, "int4", seq_len=seq, batch_size=1)
    print(f"{seq:>8} {est:>10.2f} {actual:>10.2f} {(est-actual)/actual:>7.0%}")
_ = unload(h)
```

입력이 짧을 때는 추정이 실측보다 큽니다(512 토큰에서 +7%). 그 차이는 오버헤드 15퍼센트 안에 들어옵니다. 그런데 **2048 토큰부터는 추정이 실측보다 작아지고(−17%) 4096 토큰에서는 실측의 절반에 그칩니다(−50%).** 추정이 실측보다 작은 계산기는 위험한데, 이 계산기가 입력이 길어지면 그렇게 됩니다.

원인은 가중치도 KV 캐시도 아닙니다. 가중치는 2.07GB로 고정이고 3B의 KV 캐시는 4096 토큰에 75MB뿐입니다. 실측에서 늘어난 2.8GB는 **입력을 한꺼번에 읽는 단계(프리필)에서 잠깐 생기는 중간 계산 결과**입니다. 그중 가장 큰 것이 어텐션 점수 행렬(헤드 수 × 길이 × 길이)이고 입력 길이의 제곱으로 커집니다. 1024에서 2048 토큰으로 갈 때 0.56GB, 2048에서 4096 토큰으로 갈 때 2.04GB가 늘었습니다. 길이가 두 배가 되면 네 배 가까이 늘어나는 것이 그 증거입니다. 이런 양은 비율 하나로는 맞출 수 없습니다.

오차가 음수일 때 확인 순서입니다.

1. Step 1의 검산 표에서 파라미터 오차가 1퍼센트 안쪽이었는지. 여기가 틀리면 뒤가 다 틀립니다  
2. 가중치를 선형 계층과 임베딩으로 나눠 계산했는지. 합쳐서 4bit를 곱하면 30퍼센트 가까이 과소 추정됩니다  
3. 둘 다 맞는데 입력이 길수록 오차가 커진다면 원인은 중간 계산 결과입니다. 오버헤드 비율을 올려서는 해결되지 않습니다. **운영에서 받을 최대 입력 길이로 직접 재서** 그 값을 계산기에 더해야 합니다

**측정값으로 계산기를 교정하십시오.** Step 1의 동시 사용자 표는 가중치와 KV 캐시만 더한 것이라 긴 문서를 넣는 서비스에서는 실제보다 적게 잡힙니다. 서빙 엔진(vLLM, 서빙 심화 주간)은 어텐션을 블록 단위로 계산해 이 행렬을 통째로 만들지 않으므로 같은 모델이라도 실측이 달라집니다. 계산기를 만들고 끝내면 추정이 아니라 추측입니다. 이 검증을 해 두면 용량 산정 회의에서 근거를 댈 수 있습니다.

**실행 순서** : 실습 셀 아래에 덧붙여 실행하십시오. 오차 열의 부호를 먼저 보고 어느 길이부터 음수로 바뀌는지 확인한 뒤에 크기를 보십시오.

</details>

#### 트랙 4 : 태스크별 모델 분담 설계

이론 레슨 6.2절 2단계에서 "큰 모델 하나보다 작은 모델 여러 개가 나은 경우가 많습니다"라고 했습니다. 실제로 설계해 봅니다.

In [ ]:
# 실습 : 5종 태스크를 두 모델에 분담하는 최적 조합을 찾으세요
# 목표 : 전체 정확도를 유지하면서 총 자원 소모를 최소화
# Step 3의 results[모델]["per_task"] 를 활용하세요

태스크              Qwen2.5-3B-Instruc   Llama-3.2-3B-Instr
--------------------------------------------------------
T1_요약                             O                    O
T2_추출                             O                    X
T3_분류                             O                    X
T4_규정답변                           X                    X
T5_도구선택                           O                    X

[분담안]
  T1_요약          -> Qwen2.5-3B-Instruct (2.06 GB)
  T2_추출          -> Qwen2.5-3B-Instruct (2.06 GB)
  T3_분류          -> Qwen2.5-3B-Instruct (2.06 GB)
  T4_규정답변        아무도 통과 못 함 -> 하네스 보강 또는 후보 추가
  T5_도구선택        -> Qwen2.5-3B-Instruct (2.06 GB)

<details>
<summary>정답 보기</summary>

```python
# 태스크별 모델 성적을 표로 정리합니다
ran = {n: r for n, r in results.items() if "skipped" not in r}

print(f"{'태스크':<14} " + " ".join(f"{n.split('/')[-1][:18]:>20}" for n in ran))
print("-" * (14 + 21 * len(ran)))
for t in tasks:
    row = ["O" if r["per_task"][t["id"]] else "X" for r in ran.values()]
    print(f"{t['id']:<14} " + " ".join(f"{x:>20}" for x in row))

# 분담안 : 태스크마다 통과한 모델 중 VRAM이 작은 쪽에 맡깁니다
print("\n[분담안]")
for t in tasks:
    winners = [n for n, r in ran.items() if r["per_task"][t["id"]]]
    if not winners:
        print(f"  {t['id']:<14} 아무도 통과 못 함 -> 하네스 보강 또는 후보 추가")
        continue
    pick = min(winners, key=lambda n: ran[n]["vram"])
    print(f"  {t['id']:<14} -> {pick.split('/')[-1]} ({ran[pick]['vram']:.2f} GB)")
```

**분류, 추출, 라우팅은 작은 모델로 충분하고 요약과 근거 기반 답변은 큰 모델이 유리한 경우가 많습니다.** 다만 Step 3의 두 후보는 둘 다 3B라 이 표에서는 크기 차이가 아니라 계열 차이가 보입니다. 크기 효과는 트랙 2의 7B 결과와 함께 보십시오. 어느 태스크에서 갈리는지는 태스크별 성적표로 직접 확인하십시오. 표가 있어야 "느낌상 3B가 약하다"가 아니라 "T4에서 갈린다"라고 말할 수 있습니다.

아무도 통과하지 못한 태스크가 가장 중요한 정보입니다. 모델을 더 찾기 전에 **채점기가 지나치게 빡빡하지 않은지** 먼저 보십시오. Step 2.5에서 만든 시험지에 정답 예시를 넣어 통과하는지 확인하면 됩니다.

라우팅 자체의 비용도 잊지 마십시오. 질문을 분류하려고 모델을 한 번 더 호출하면 그것도 자원이므로 규칙 기반 라우팅으로 처리할 수 있으면 그쪽이 낫습니다.

**실행 순서** : Step 3을 먼저 돌려 `results`가 채워져 있어야 합니다. 그다음 이 블록을 실습 셀 아래에 덧붙여 실행하십시오.

</details>

#### 트랙 5 : 양자화 정밀도와 품질

지금까지 모든 모델을 4bit로 올렸습니다. 그 대가를 잽니다.

In [31]:
def load_with_precision(name, precision):
    kwargs = {"device_map": "auto"}
    if precision == "int4":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    elif precision == "int8":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        kwargs["dtype"] = torch.float16
    else:
        kwargs["dtype"] = torch.float16

    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, **kwargs)
    model.eval()
    return {"repo": name, "tok": tok, "model": model}


NAME = models["Qwen/Qwen2.5-3B-Instruct"]["repo"]
cols = [t["id"].split("_")[0] for t in tasks]
rows = {}
failed_logs = []  # 🔥 실패한 오답들을 모아둘 리스트

for p in ["int4", "int8", "fp16"]:
    try:
        h = load_with_precision(NAME, p)
    except Exception as e:
        rows[p] = {"error": type(e).__name__}
        print(f"{p} 로드 실패 : {type(e).__name__}")
        continue

    vram = vram_gb()
    per_task, speeds = {}, []

    for t in tasks:
        ans, tps, _ = generate(h["tok"], h["model"], t["prompt"])
        ok = bool(t["check"](ans))
        per_task[t["id"]] = ok
        speeds.append(tps)

        # 🔥 [개선점] X가 뜬 경우, 실제 답변을 기록해 둠
        if not ok:
            failed_logs.append({
                "precision": p,
                "task": t["id"],
                "answer": ans.strip(),
            })

    rows[p] = {
        "per_task": per_task,
        "tps": float(np.mean(speeds)),
        "vram": vram,
    }
    unload(h)

# 1. 태스크별 비교표 출력
print(
    f"\n{'정밀도':>8} "
    + " ".join(f"{c:>4}" for c in cols)
    + f" {'평균':>7} {'tok/s':>7} {'VRAM':>7}"
)
print("-" * 62)
for p, r in rows.items():
    if "error" in r:
        print(f"{p:>8} {'로드 실패 : ' + r['error']:>44}")
        continue
    marks = " ".join(
        f"{'O' if r['per_task'][t['id']] else 'X':>4}" for t in tasks
    )
    acc = sum(r["per_task"].values()) / len(r["per_task"])
    print(f"{p:>8} {marks} {acc:>6.0%} {r['tps']:>7.1f} {r['vram']:>6.2f}G")

# 2. 🔥 [개선점] 오답 원인 분석 (왜 X가 떴는가?)
if failed_logs:
    print("\n" + "=" * 50)
    print("🚨 [품질 저하 분석] 오답을 낸 답변 원문 모아보기")
    print("=" * 50)
    for log in failed_logs:
        print(f"[{log['precision'].upper()} | {log['task']}]")
        print(f"-> 모델 답변:\n{log['answer'][:200]}...")  # 앞부분 200자 출력
        print("-" * 50)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  [해제] VRAM 2.07 -> 0.01 GB  (줄어든 양 2.07)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  [해제] VRAM 3.44 -> 0.01 GB  (줄어든 양 3.43)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  [해제] VRAM 6.18 -> 0.01 GB  (줄어든 양 6.17)

     정밀도   T1   T2   T3   T4   T5      평균   tok/s    VRAM
--------------------------------------------------------------
    int4    O    O    O    X    O    80%    11.9   2.07G
    int8    O    O    O    X    O    80%     5.0   3.44G
    fp16    O    O    O    O    O   100%    18.8   6.18G

🚨 [품질 저하 분석] 오답을 낸 답변 원문 모아보기
[INT4 | T4_규정답변]
-> 모델 답변:
물론입니다. 규정에 따르면 개인 사정으로 연차를 사용하지 못한 경우, 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 연차 휴가를 이월할 수 있습니다. 따라서 개인 사정으로 연차를 못 쓰셨다면 인사팀장의 승인이 필요합니다. 이를 받으시면 내년으로 연차 휴가를 이월할 수 있습니다. 제7조 (연차 휴가의 이월) 참조....
--------------------------------------------------
[INT8 | T4_규정답변]
-> 모델 답변:
제7조 (연차 휴가의 이월)에 따르면, 개인 사정으로 연차를 사용하지 못한 경우에도 회사의 업무 상황이 불가피한 사유가 있는지 확인 후 인사팀장의 승인이 필요합니다. 따라서 개인 사정으로 연차를 사용하지 못했더라도 회사의 업무 상황이 불가피한 사유가 있는지 확인하고 인사팀장의 승인이 필요합니다. (제7조)...
--------------------------------------------------


<details>
<summary>정답 보기</summary>

```python
def load_with_precision(name, precision):
    kwargs = {"device_map": "auto"}
    if precision == "int4":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    elif precision == "int8":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        kwargs["dtype"] = torch.float16      # 양자화되지 않는 층과 입력을 fp16으로. bf16이면 곱셈마다 변환 경고가 찍힙니다
    else:
        kwargs["dtype"] = torch.float16      # torch_dtype= 은 폐기되었습니다
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, **kwargs)
    model.eval()
    return {"repo": name, "tok": tok, "model": model}

NAME = models["Qwen/Qwen2.5-3B-Instruct"]["repo"]
cols = [t["id"].split("_")[0] for t in tasks]
rows = {}

for p in ["int4", "int8", "fp16"]:
    try:
        h = load_with_precision(NAME, p)
    except Exception as e:
        rows[p] = {"error": type(e).__name__}
        print(f"{p} 로드 실패 : {type(e).__name__}")
        continue
    vram = vram_gb()
    per_task, speeds = {}, []
    for t in tasks:
        ans, tps, _ = generate(h["tok"], h["model"], t["prompt"])
        per_task[t["id"]] = bool(t["check"](ans))
        speeds.append(tps)
    rows[p] = {"per_task": per_task, "tps": float(np.mean(speeds)), "vram": vram}
    unload(h)

# 태스크별 칸을 남긴 비교표. 평균 한 줄로는 어디가 무너졌는지 알 수 없습니다
print(f"\n{'정밀도':>8} " + " ".join(f"{c:>4}" for c in cols)
      + f" {'평균':>7} {'tok/s':>7} {'VRAM':>7}")
print("-" * 62)
for p, r in rows.items():
    if "error" in r:
        print(f"{p:>8} {'로드 실패 : ' + r['error']:>44}")
        continue
    marks = " ".join(f"{'O' if r['per_task'][t['id']] else 'X':>4}" for t in tasks)
    acc = sum(r["per_task"].values()) / len(r["per_task"])
    print(f"{p:>8} {marks} {acc:>6.0%} {r['tps']:>7.1f} {r['vram']:>6.2f}G")
```

**태스크별 칸이 이 트랙의 전부입니다.** 평균만 보면 'int4와 int8이 80%, fp16이 100%'로 끝나는데 칸을 나누면 **떨어진 곳이 T4 하나**라는 것이 보입니다.

이번 실행에서 양자화가 먼저 망가뜨린 것은 형식 출력(T2)이 아니라 **규정을 읽고 원칙과 예외를 구분하는 T4**였습니다. int4에서는 Step 3과 같은 답(개인 사정도 승인받으면 이월 가능)이 나왔고 int8도 같은 칸에서 떨어졌으며 fp16에서만 맞혔습니다. 이론 레슨 3.6절이 약하다고 한 '여러 단계로 추론하기' 쪽입니다. 이 표를 보면 Step 3의 결과도 다시 봐야 합니다. Step 3에서 3B가 T4를 틀린 것은 모델 크기의 한계가 아니라 **4비트의 대가**였습니다. 이론 레슨 6.5절의 퀴즈가 말한 '모델을 키우기 전에 양자화를 의심하라'를 보여 주는 결과입니다.

속도 열도 보십시오. 이 환경에서는 fp16이 19.6 tok/s로 가장 빠르고 int4가 12.5, int8이 5.0입니다. 이론 레슨 3.3절의 '양자화하면 대체로 빨라진다'와 반대인데, bitsandbytes의 양자화는 곱셈 때마다 가중치를 다시 풀어 쓰는 방식이라 혼자 쓰는 작은 배치에서는 푸는 시간이 아낀 시간보다 큽니다. 8비트 방식(LLM.int8)은 이상치 열을 따로 계산하는 분해까지 더해져 더 느립니다. 메모리는 int4 2.07GB, int8 3.44GB, fp16 6.18GB로 비트 수에 거의 비례합니다. 그래서 이 환경에서 int8은 '품질은 int4와 같고 속도는 절반 이하'라 후보에서 빠지고, 실제 선택은 **int4에 T4를 보완할 하네스(트랙 2)를 붙이는 것**과 **fp16이 올라가는 더 큰 GPU** 둘 중 하나입니다.

GPTQ와 AWQ 형식은 이론 레슨에서 봤고 서빙 엔진과 함께 쓰는 일은 서빙 심화 주간에서 다룹니다. 그 형식들은 전용 커널로 풀기와 곱셈을 한 번에 하므로 양자화가 실제로 빨라지는 것은 거기서 확인할 수 있습니다.

**실행 순서** : 이 블록을 실습 셀 아래에 덧붙여 실행하십시오. 모델을 세 번 올리므로 12~18분이 걸립니다. fp16에서 메모리 부족이 나면 그 줄만 "로드 실패"로 찍히고 나머지는 그대로 진행됩니다.

</details>

## Step 8 : 공유와 정리

각자 수행한 두 트랙과 설계서를 공유합니다.

**1. 선택한 트랙과 측정 결과**

**2. 내가 고른 모델과 근거** : 라이선스, VRAM, 평가 결과를 모두 인용해 설명

**3. 설계서 피어 리뷰** : 옆 사람의 설계서를 읽고 아래를 확인
- 프라이버시 경계가 기술 선택보다 먼저 정해졌는가  
- 모델 선정 근거에 측정 결과가 인용되었는가  
- 성공 기준이 측정 가능한 형태인가  
- RAG를 먼저 시도하는 순서가 반영되었는가

**4. 리뷰받은 내용 중 수정할 점**

이론 레슨 6.2절의 **자르는 네 단계**를 코드로 옮겼습니다. **경계**를 "우리 GPU 안"으로 고정해 후보를 오픈 웨이트로 한정했고 **태스크**를 다섯 종으로 적어 무엇을 잴지 정했습니다. **라이선스** 체크리스트로 법무에 넘길 항목을 추렸고 **메모리** 계산기로 올라가는지와 서비스가 되는지를 나눠 판단했습니다.

남은 후보 중에서 고르는 단계는 **우리 평가 세트로 직접 재는 것** 하나뿐이었습니다. 리더보드 점수보다 우리 태스크 측정이 정확하다는 것은 프라이빗 RAG 노드에서 임베딩 모델을 고를 때와 같은 원칙입니다.

또 **재기 전에 자를 먼저 검산했습니다.** 채점기의 `"없" in s` 하나가 정반대 답을 정답으로 세고 있었고 그대로 두었다면 우리는 모델을 탓했을 것입니다. 측정 도구의 버그는 모델의 한계로 위장합니다.

여기까지는 주어진 모델을 써 왔습니다. 검색으로 지식을 공급하고 하네스로 형식을 잡고 이미지를 처리했습니다.

다음 노드에서 처음으로 사내 데이터로 파인튜닝해 **모델 자체를 바꿉니다.** 오늘 쓴 설계서가 그 작업의 기준입니다.

## 마무리 과제 : 모델 선정 보고서

여러분이 가정한 사내 프로젝트에 쓸 모델을 선정하고 보고서를 작성하십시오. 실무에서 그대로 제출할 수 있는 형태로 만듭니다.

**1단계. 요구사항 정의**

1. 자동화할 업무를 정합니다. 사용자 수, 동시 사용자 수, 목표 응답 시간을 숫자로 적습니다.  
2. 그 업무에서 모델이 수행할 태스크를 3종 이상 정의합니다.

**2단계. 후보 선정과 라이선스 필터**

3. 오픈 웨이트 후보 4개 이상을 나열합니다.  
4. 라이선스 체크리스트 6항목에 **게이트 여부**를 더해 표를 완성합니다.  
5. 라이선스로 탈락한 후보와 사유를 명시합니다.

**3단계. VRAM 계산**

6. 남은 후보에 대해 정밀도별 VRAM을 계산합니다.  
7. 목표 동시 사용자 수와 컨텍스트 길이를 반영해 실제 운영 가능 여부를 판단합니다.  
8. 추정치를 실측으로 검증합니다.

**4단계. 자체 평가**

9. 태스크별 평가 세트를 만듭니다. 태스크당 4문항 이상.  
10. **채점기를 먼저 검산합니다.** 태스크마다 정반대 답과 빈 문자열을 넣어 걸러지는지 확인하고 그 결과를 보고서에 한 줄 남깁니다.  
11. 후보 모델을 측정해 표를 완성합니다.

| 모델 | 태스크1 | 태스크2 | 태스크3 | TTFT | TPS | VRAM | 게이트 |
|---|---|---|---|---|---|---|---|

**5단계. 보고서**

- **최종 선정 모델과 근거** : 라이선스, VRAM, 평가 결과를 모두 인용  
- **탈락한 후보와 사유** : 각 후보가 어느 단계에서 왜 탈락했는지. 받지 못해 건너뛴 후보도 그 사유로 적습니다  
- **하네스로 보완할 부분** : 선정 모델이 약한 태스크를 어떻게 보완할지  
- **법무 확인 요청 항목** : 라이선스에서 판단이 불확실한 부분  
- **재검토 조건** : 어떤 상황이 되면 모델을 다시 고를지

**평가 기준**

특정 모델을 골랐는지가 아니라 **선정 과정이 재현 가능한지**로 평가합니다. 다른 사람이 같은 절차를 따라가면 같은 결론에 도달할 수 있어야 하고 조건이 바뀌면 결론이 어떻게 바뀔지 설명할 수 있어야 합니다.